# theLook Analytics · 03 · Funnel, attribution e A/B test

theLook non contiene esperimenti, e il suo generatore decide funnel e canali con regole fisse (vedi il README). Per le domande 2 (funnel), 1 (attribution) e 5 (A/B test) questo notebook usa un secondo dataset: [E-Commerce Clickstream Simulation Dataset](https://www.kaggle.com/datasets/amith1707/e-commerce-clickstream-simulation-dataset) su Kaggle, licenza CC0. Sono 5,77 milioni di sessioni simulate su cinque marketplace, dal 2023 al 2024, con 6 esperimenti A/B.

- **Funnel:** dove si perdono le sessioni tra carrello, checkout e acquisto, e se cambia per canale, browser, dispositivo o piattaforma.
- **Attribution:** quanto vale ciascun canale secondo cinque modelli, dal last click alle catene di Markov.
- **A/B test:** controllo dell'assegnazione, potenza statistica, lettura frequentista e bayesiana, effetti per segmento e nel tempo.

Tutto gira in Google Colab: i file restano sul disco di Colab e si interrogano in SQL con **DuckDB**, senza caricarli in memoria.

**Come eseguirlo**: *Runtime → Esegui tutto*. Ci vogliono circa 10 minuti, quasi tutti per scaricare e caricare i dati.

## 1 · Download e caricamento

`kagglehub` scarica i dataset pubblici di Kaggle senza account. Se chiede le credenziali, esegui `kagglehub.login()` in una cella nuova e inserisci tu nome utente e chiave API di Kaggle.

Si carica solo il file completo (`data/full`), una riga per sessione, con le colonne che servono. Gli altri file del dataset ne sono sottoinsiemi: per esempio `clickstream_experiments` contiene le 4.264.236 sessioni con almeno un esperimento.

In [ ]:
import os
import re

try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

path = kagglehub.dataset_download("amith1707/e-commerce-clickstream-simulation-dataset")
print("Cartella:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import duckdb
except ImportError:
    !pip install -q duckdb
    import duckdb

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

ACCENTO, GRIGIO, SCURO = "#118DFF", "#C8C8C8", "#555555"
Z = stats.norm.ppf(0.975)       # livello di confidenza 95%
Z_POT = stats.norm.ppf(0.80)    # potenza 80%

con = duckdb.connect("/content/clickstream.duckdb")


def q(sql):
    return con.sql(sql).df()


def esiste(tabella):
    sql = f"SELECT COUNT(*) FROM information_schema.tables WHERE table_name = '{tabella}'"
    return con.sql(sql).fetchone()[0] > 0


def pct(x, d=1):
    return "–" if pd.isna(x) else f"{x * 100:.{d}f}%".replace(".", ",")


def num(x, d=0):
    if pd.isna(x):
        return "–"
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def wilson(k, n):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    den = 1 + Z**2 / n
    centro = (p + Z**2 / (2 * n)) / den
    mezzo = Z * np.sqrt(p * (1 - p) / n + Z**2 / (4 * n**2)) / den
    return centro - mezzo, centro + mezzo


def holm(p):
    p = np.asarray(p, float)
    m, corretti, massimo = len(p), np.empty(len(p)), 0.0
    for rango, i in enumerate(np.argsort(p)):
        massimo = max(massimo, min(1.0, (m - rango) * p[i]))
        corretti[i] = massimo
    return corretti


COLONNE = """
    session_id, user_id_hash AS user_id, session_start, platform, device_type, browser,
    traffic_source, landing_page_type, had_cart_event, checkout_started, had_purchase,
    COALESCE(session_revenue_usd, 0) AS ricavi, is_bot_suspect,
    customer_segment, user_archetype, primary_device, country,
    enrolled_experiments, enrolled_variants
"""

if not esiste("sessioni_tutte"):
    # Se il file completo è già nel database (esecuzione precedente nella stessa sessione), si parte da lì
    file_completo = os.path.join(path, "data", "full", "clickstream_full.csv")
    origine = "clickstream_full" if esiste("clickstream_full") else f"read_csv_auto('{file_completo}')"
    con.sql(f"CREATE TABLE sessioni_tutte AS SELECT {COLONNE} FROM {origine}")

print(num(con.sql("SELECT COUNT(*) FROM sessioni_tutte").fetchone()[0]), "sessioni caricate")

## 2 · Il dataset e i suoi limiti

L'esplorazione (prima versione di questo notebook) ha confrontato la documentazione del dataset con i dati:

- **Periodo e copertura:** 1 gennaio 2023 – 31 dicembre 2024, circa 500.000 utenti, 5 marketplace (Amazon, eBay, Walmart, Flipkart, BestBuy).
- **Documentazione e dati non coincidono:** conversione 0,78% contro lo 0,9% dichiarato; rimbalzi 2,3% contro 3,7%; 11 paesi contro 10 (c'è anche il Messico); due segnali descritti, `fake_discount_exposure` e `post_purchase_regret_flag`, sono sempre falsi. Valgono i dati.
- **Bot:** 121.242 sessioni (2,1%) sono segnalate come sospette. Si escludono da tutte le analisi.
- **Dati sintetici:** come per theLook, i risultati mostrano il metodo. Alcuni effetti possono essere stati scritti nel generatore.

La cella seguente ripete i controlli principali e crea la tabella `sessioni` su cui lavorano tutte le analisi: senza bot e senza sessioni duplicate.

In [ ]:
controllo = q("""
SELECT COUNT(*) AS righe,
       COUNT(DISTINCT session_id) AS sessioni_distinte,
       COUNT(DISTINCT user_id) AS utenti,
       MIN(session_start)::DATE AS dal,
       MAX(session_start)::DATE AS al,
       AVG(had_purchase::INTEGER) AS conversione,
       SUM(is_bot_suspect::INTEGER) AS sessioni_bot,
       AVG(CASE WHEN is_bot_suspect THEN had_purchase::INTEGER END) AS conversione_bot,
       AVG(CASE WHEN NOT is_bot_suspect THEN had_purchase::INTEGER END) AS conversione_senza_bot
FROM sessioni_tutte
""")
print(controllo.T.to_string(header=False), "\n")

duplicati = q("""
SELECT COUNT(*) AS id_ripetuti,
       COALESCE(SUM(n), 0) AS righe_coinvolte,
       COALESCE(SUM((utenti > 1)::INTEGER), 0) AS id_con_utenti_diversi
FROM (SELECT session_id, COUNT(*) AS n, COUNT(DISTINCT user_id) AS utenti
      FROM sessioni_tutte GROUP BY session_id HAVING COUNT(*) > 1)
""")
print(duplicati.T.to_string(header=False))

con.sql("""
CREATE OR REPLACE TABLE sessioni AS
SELECT * FROM sessioni_tutte
WHERE NOT is_bot_suspect
QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id, user_id ORDER BY session_start) = 1
""")
print("\nSessioni analizzate, senza bot e senza duplicati:", num(con.sql("SELECT COUNT(*) FROM sessioni").fetchone()[0]))

In [ ]:
# Cosa dice la documentazione del dataset sugli esperimenti
readme = open(os.path.join(path, "README.md"), encoding="utf-8").read()
for sezione in re.split(r"\n(?=#{2,4} )", readme):
    titolo = sezione.strip().splitlines()[0] if sezione.strip() else ""
    if re.search(r"experiment|a/b", titolo, re.IGNORECASE):
        print(sezione.strip()[:4000], "\n")

## 3 · Funnel (domanda 2)

Ogni sessione ha tre passaggi: aggiunta al carrello, inizio del checkout, acquisto. Prima il funnel complessivo, poi per canale, browser, dispositivo e piattaforma.

Con milioni di sessioni anche differenze minime risultano statisticamente significative, quindi il p-value dice poco. Per capire se una differenza conta si usa la **V di Cramér**, che misura la forza dell'associazione: sotto 0,1 è trascurabile.

In [ ]:
f = q("""
SELECT COUNT(*) AS sessioni,
       SUM(had_cart_event::INTEGER) AS carrello,
       SUM(checkout_started::INTEGER) AS checkout,
       SUM(had_purchase::INTEGER) AS acquisto,
       SUM((checkout_started AND NOT had_cart_event)::INTEGER) AS checkout_senza_carrello,
       SUM((had_purchase AND NOT checkout_started)::INTEGER) AS acquisto_senza_checkout
FROM sessioni
""").iloc[0]
print("Coerenza dei passaggi: checkout senza carrello", num(f["checkout_senza_carrello"]),
      "· acquisti senza checkout", num(f["acquisto_senza_checkout"]), "\n")

funnel = pd.DataFrame({"fase": ["Sessioni", "Carrello", "Checkout", "Acquisto"],
                       "n": [float(f[c]) for c in ["sessioni", "carrello", "checkout", "acquisto"]]})
funnel["sul totale"] = funnel["n"] / funnel["n"].iloc[0]
funnel["sulla fase precedente"] = funnel["n"] / funnel["n"].shift(1)
funnel["persi"] = funnel["n"].shift(1) - funnel["n"]
display(pd.DataFrame({
    "fase": funnel["fase"],
    "sessioni": funnel["n"].map(num),
    "% sul totale": funnel["sul totale"].map(lambda x: pct(x, 2)),
    "% sulla fase precedente": funnel["sulla fase precedente"].map(pct),
    "persi nel passaggio": funnel["persi"].map(num),
}))

In [ ]:
PASSAGGI = [("carrello", "sessioni", "Carrello su sessioni"),
            ("checkout", "carrello", "Checkout su carrello"),
            ("acquisto", "checkout", "Acquisto su checkout"),
            ("acquisto", "sessioni", "Conversione totale")]
DIMENSIONI = {"traffic_source": "Canale", "browser": "Browser",
              "device_type": "Dispositivo", "platform": "Piattaforma"}


def funnel_per(dim):
    d = q(f"""
    SELECT {dim} AS valore, COUNT(*) AS sessioni,
           SUM(had_cart_event::INTEGER) AS carrello,
           SUM(checkout_started::INTEGER) AS checkout,
           SUM(had_purchase::INTEGER) AS acquisto
    FROM sessioni GROUP BY 1 ORDER BY 2 DESC
    """)
    for numeratore, denominatore, nome in PASSAGGI:
        d[nome] = d[numeratore] / d[denominatore]
    return d


def cramer_v(successi, totali):
    successi, totali = np.asarray(successi, float), np.asarray(totali, float)
    tabella = np.vstack([successi, totali - successi])
    chi2, p, _, _ = stats.chi2_contingency(tabella, correction=False)
    return np.sqrt(chi2 / tabella.sum()), p


funnel_dim, sintesi = {}, []
for dim, etichetta in DIMENSIONI.items():
    d = funnel_per(dim)
    funnel_dim[dim] = d
    print(f"\n{etichetta}")
    display(pd.DataFrame({
        etichetta: d["valore"],
        "sessioni": d["sessioni"].map(num),
        **{nome: d[nome].map(lambda x: pct(x, 2)) for _, _, nome in PASSAGGI},
    }))
    for numeratore, denominatore, nome in PASSAGGI:
        v, p = cramer_v(d[numeratore], d[denominatore])
        tassi = d[nome]
        sintesi.append({"dimensione": etichetta, "passaggio": nome,
                        "V di Cramér": round(v, 4), "p": f"{p:.1e}",
                        "più basso": f"{d['valore'][tassi.idxmin()]} {pct(tassi.min(), 2)}",
                        "più alto": f"{d['valore'][tassi.idxmax()]} {pct(tassi.max(), 2)}",
                        "alto / basso": round(tassi.max() / tassi.min(), 2)})

print("\nQuanto contano le differenze (V di Cramér: sotto 0,1 trascurabile)")
sintesi_funnel = pd.DataFrame(sintesi)
display(sintesi_funnel)

### Dove si perde di più

L'unica differenza che conta è tra piattaforme, nel passaggio dal carrello al checkout. Le celle seguenti la mostrano, verificano se dipende da un dispositivo, un canale, un paese o un periodo, e stimano quanto vale.

In [ ]:
piattaforme = funnel_dim["platform"]
PEGGIORE = piattaforme.loc[piattaforme["Checkout su carrello"].idxmin(), "valore"]
altre = piattaforme[piattaforme["valore"] != PEGGIORE]
tasso_peggiore = piattaforme.loc[piattaforme["valore"] == PEGGIORE, "Checkout su carrello"].iloc[0]
tasso_altre = altre["checkout"].sum() / altre["carrello"].sum()

d = piattaforme.sort_values("Checkout su carrello")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(d["valore"], d["Checkout su carrello"] * 100,
        color=[ACCENTO if v == PEGGIORE else GRIGIO for v in d["valore"]])
for y, v in enumerate(d["Checkout su carrello"]):
    ax.text(v * 100 + 0.5, y, pct(v), va="center", fontsize=10)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Su {PEGGIORE} arriva al checkout il {pct(tasso_peggiore, 0)} dei carrelli, "
             f"sulle altre piattaforme il {pct(tasso_altre, 0)}", loc="left", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Il calo dipende da un dispositivo, un canale, un paese o un periodo?
SCOMPOSIZIONI = {"device_type": "Dispositivo", "traffic_source": "Canale", "country": "Paese",
                 "date_trunc('quarter', session_start)::DATE": "Trimestre"}
for espressione, etichetta in SCOMPOSIZIONI.items():
    d = q(f"""
    SELECT {espressione} AS valore,
           SUM(CASE WHEN platform = '{PEGGIORE}' THEN 1 ELSE 0 END) AS carrelli,
           SUM(CASE WHEN platform = '{PEGGIORE}' THEN checkout_started::INTEGER END)
             / SUM(CASE WHEN platform = '{PEGGIORE}' THEN 1 END) AS tasso_peggiore,
           SUM(CASE WHEN platform <> '{PEGGIORE}' THEN checkout_started::INTEGER END)
             / SUM(CASE WHEN platform <> '{PEGGIORE}' THEN 1 END) AS tasso_altre
    FROM sessioni WHERE had_cart_event
    GROUP BY 1 ORDER BY 1
    """)
    print(f"\nCheckout su carrello per {etichetta.lower()}")
    display(pd.DataFrame({
        etichetta: d["valore"].astype(str),
        f"carrelli su {PEGGIORE}": d["carrelli"].map(num),
        PEGGIORE: d["tasso_peggiore"].map(pct),
        "altre piattaforme": d["tasso_altre"].map(pct),
        "rapporto": (d["tasso_peggiore"] / d["tasso_altre"]).map(lambda x: num(x, 2)),
    }))

In [ ]:
# Quanto vale il problema: acquisti e ricavi in più se il passaggio al checkout fosse come sulle altre piattaforme
valore = q(f"""
SELECT SUM(had_cart_event::INTEGER) AS carrelli,
       AVG(CASE WHEN checkout_started THEN had_purchase::INTEGER END) AS acquisto_su_checkout,
       AVG(CASE WHEN had_purchase THEN ricavi END) AS ricavo_medio_acquisto,
       SUM(ricavi) AS ricavi
FROM sessioni WHERE platform = '{PEGGIORE}'
""").iloc[0]
checkout_in_piu = valore["carrelli"] * (tasso_altre - tasso_peggiore)
acquisti_in_piu = checkout_in_piu * valore["acquisto_su_checkout"]
ricavi_in_piu = acquisti_in_piu * valore["ricavo_medio_acquisto"]
print(f"{PEGGIORE}, 2023–2024: {num(valore['carrelli'])} carrelli, ricavi ${num(valore['ricavi'])}")
print(f"Con il passaggio al checkout delle altre piattaforme ({pct(tasso_altre)} invece di {pct(tasso_peggiore)}):")
print(f"  +{num(checkout_in_piu)} checkout, +{num(acquisti_in_piu)} acquisti, "
      f"+${num(ricavi_in_piu)} di ricavi in due anni ({pct(ricavi_in_piu / valore['ricavi'], 0)} in più)")

## 4 · Attribution (domanda 1)

Prima di scegliere un modello si controlla se i dati contengono un segnale da attribuire:

1. **Il canale cambia la probabilità di acquisto?** Se tutte le sessioni convertono allo stesso modo, qualunque canale si premi, il merito finisce in proporzione al traffico.
2. **Il canale di una sessione dipende da quello della precedente?** Se i percorsi sono casuali, i modelli multi-touch non hanno struttura su cui lavorare (è quello che succede in theLook).

Poi si costruiscono i **percorsi**: per ogni utente, le sessioni dalla prima (o dal giorno dopo l'acquisto precedente) fino all'acquisto. Su questi percorsi si confrontano cinque modelli: last click, first click, lineare, a U (40% alla prima sessione, 40% all'ultima, 20% alle altre) e **catene di Markov**, che misurano quanto calano le conversioni se si toglie un canale dai percorsi (*removal effect*).

In [ ]:
# 1. Conversione per canale, con intervallo di confidenza
conv = funnel_dim["traffic_source"]
basso, alto = wilson(conv["acquisto"], conv["sessioni"])
display(pd.DataFrame({"canale": conv["valore"], "sessioni": conv["sessioni"].map(num),
                      "conversione": conv["Conversione totale"].map(lambda x: pct(x, 2)),
                      "IC 95%": [f"{pct(b, 2)} – {pct(a, 2)}" for b, a in zip(basso, alto)]}))

# 2. Canale di una sessione rispetto a quello della sessione precedente dello stesso utente
coppie = q("""
WITH s AS (
  SELECT traffic_source AS canale,
         LAG(traffic_source) OVER (PARTITION BY user_id ORDER BY session_start, session_id) AS precedente
  FROM sessioni
)
SELECT precedente, canale, COUNT(*) AS n
FROM s WHERE precedente IS NOT NULL
GROUP BY 1, 2
""")
tabella = coppie.pivot(index="precedente", columns="canale", values="n").fillna(0)
chi2, p, _, _ = stats.chi2_contingency(tabella.values)
v = np.sqrt(chi2 / (tabella.values.sum() * (min(tabella.shape) - 1)))
print(f"\nCanale della sessione e canale della precedente: p = {p:.2g}, V di Cramér = {v:.3f}")
print("Canale della sessione successiva, per canale della precedente (% di riga):")
display((tabella.div(tabella.sum(axis=1), axis=0) * 100).round(1))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE percorsi AS
WITH s AS (
  SELECT user_id, session_id, session_start, traffic_source AS canale, had_purchase,
         -- numero di acquisti precedenti: un nuovo percorso inizia dopo ogni acquisto
         COALESCE(SUM(had_purchase::INTEGER) OVER (
           PARTITION BY user_id ORDER BY session_start, session_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS percorso
  FROM sessioni
)
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY user_id, percorso ORDER BY session_start, session_id) AS passo,
       COUNT(*) OVER (PARTITION BY user_id, percorso) AS lunghezza,
       MAX(had_purchase::INTEGER) OVER (PARTITION BY user_id, percorso) AS convertito
FROM s
""")

riepilogo = q("""
SELECT COUNT(*) AS percorsi,
       SUM(convertito) AS percorsi_con_acquisto,
       AVG(CASE WHEN convertito = 1 THEN lunghezza END) AS sessioni_medie_fino_all_acquisto,
       AVG(CASE WHEN convertito = 1 THEN (lunghezza = 1)::INTEGER END) AS quota_acquisti_alla_prima_sessione,
       AVG(CASE WHEN convertito = 1 THEN (canali = 1)::INTEGER END) AS quota_percorsi_con_un_solo_canale
FROM (SELECT user_id, percorso, ANY_VALUE(lunghezza) AS lunghezza, ANY_VALUE(convertito) AS convertito,
             COUNT(DISTINCT canale) AS canali
      FROM percorsi GROUP BY user_id, percorso)
""")
print(riepilogo.T.to_string(header=False))

regole = q("""
SELECT canale,
       SUM(CASE WHEN passo = lunghezza THEN 1 ELSE 0 END) AS "Last click",
       SUM(CASE WHEN passo = 1 THEN 1 ELSE 0 END) AS "First click",
       SUM(1 / lunghezza) AS "Lineare",
       SUM(CASE WHEN lunghezza = 1 THEN 1.0::DOUBLE
                WHEN lunghezza = 2 THEN 0.5::DOUBLE
                WHEN passo = 1 OR passo = lunghezza THEN 0.4::DOUBLE
                ELSE 0.2::DOUBLE / (lunghezza - 2) END) AS "A U"
FROM percorsi
WHERE convertito = 1
GROUP BY canale
""")
traffico = q("SELECT traffic_source AS canale, COUNT(*) AS sessioni FROM sessioni GROUP BY 1")

In [ ]:
# Catene di Markov: transizioni tra canali dentro ogni percorso, dall'inizio fino ad acquisto o abbandono
transizioni = q("""
WITH t AS (
  SELECT COALESCE(LAG(canale) OVER (PARTITION BY user_id, percorso ORDER BY passo), 'start') AS da,
         canale AS a
  FROM percorsi
)
SELECT da, a, COUNT(*) AS n FROM t GROUP BY 1, 2
UNION ALL
SELECT canale AS da, CASE WHEN convertito = 1 THEN 'conv' ELSE 'null' END AS a, COUNT(*) AS n
FROM percorsi WHERE passo = lunghezza GROUP BY 1, 2
""")


def prob_conversione(transizioni, rimosso=None):
    t = transizioni.copy()
    if rimosso is not None:
        # Senza il canale, chi ci sarebbe passato abbandona il percorso
        t = t[t["da"] != rimosso].copy()
        t.loc[t["a"] == rimosso, "a"] = "null"
    stati = sorted((set(t["da"]) | set(t["a"])) - {"conv", "null"})
    indice = {s: i for i, s in enumerate(stati)}
    Q, R = np.zeros((len(stati), len(stati))), np.zeros((len(stati), 2))
    for da, a, n in t.groupby(["da", "a"], as_index=False)["n"].sum().itertuples(index=False):
        if a == "conv":
            R[indice[da], 0] += n
        elif a == "null":
            R[indice[da], 1] += n
        else:
            Q[indice[da], indice[a]] += n
    totali = Q.sum(axis=1) + R.sum(axis=1)
    totali[totali == 0] = 1
    Q, R = Q / totali[:, None], R / totali[:, None]
    assorbimento = np.linalg.solve(np.eye(len(stati)) - Q, R)
    return assorbimento[indice["start"], 0]


p_base = prob_conversione(transizioni)
canali = sorted(traffico["canale"])
removal = {c: 1 - prob_conversione(transizioni, c) / p_base for c in canali}
print(f"Probabilità di acquisto di un percorso: {pct(p_base, 2)}")
print("Removal effect (calo delle conversioni senza il canale):",
      ", ".join(f"{c} {pct(r)}" for c, r in sorted(removal.items(), key=lambda x: -x[1])), "\n")

conversioni = regole["Last click"].sum()
regole["Markov"] = regole["canale"].map(lambda c: removal[c] / sum(removal.values()) * conversioni)

MODELLI = ["Last click", "First click", "Lineare", "A U", "Markov"]
attribution = traffico.merge(regole, on="canale")
attribution["Quota di sessioni"] = attribution["sessioni"] / attribution["sessioni"].sum()
for m in MODELLI:
    attribution[m] = attribution[m] / attribution[m].sum()
attribution = attribution.sort_values("Last click", ascending=False).set_index("canale")

print("Quota di conversioni attribuita a ogni canale")
display(attribution[["Quota di sessioni"] + MODELLI].apply(lambda col: col.map(pct)))

scarti = attribution[MODELLI].sub(attribution["Quota di sessioni"], axis=0)
print(f"Scarto massimo tra un modello e la quota di sessioni: {pct(scarti.abs().max().max())}")
print(f"Scarto massimo tra Markov e last click: {pct((attribution['Markov'] - attribution['Last click']).abs().max())}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
y = np.arange(len(attribution))
altezza = 0.27
ax.barh(y - altezza, attribution["Quota di sessioni"] * 100, altezza, color=GRIGIO, label="Quota di sessioni")
ax.barh(y, attribution["Last click"] * 100, altezza, color=SCURO, label="Last click")
ax.barh(y + altezza, attribution["Markov"] * 100, altezza, color=ACCENTO, label="Markov")
ax.set_yticks(y, attribution.index)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Quota di conversioni per canale: last click e Markov a confronto con il traffico", loc="left", fontsize=12)
plt.tight_layout()
plt.show()

## 5 · A/B test (domanda 5)

Ogni sessione può partecipare a più esperimenti: le colonne `enrolled_experiments` e `enrolled_variants` contengono due liste allineate, separate da `|`. Si trasformano in una riga per sessione ed esperimento.

Prima dei risultati si controlla il disegno:

- **Unità di assegnazione:** se ogni utente resta nella stessa variante per tutte le sue sessioni, l'unità di analisi è l'utente, non la sessione. Contare le sessioni di uno stesso utente come osservazioni indipendenti renderebbe gli intervalli di confidenza troppo stretti.
- **Sample ratio mismatch (SRM):** se le varianti hanno numerosità diverse da quelle previste, l'assegnazione ha un problema e i risultati non sono affidabili. Si verifica sugli esperimenti a due varianti, che dovrebbero essere divisi a metà. Per quelli a quattro varianti la ripartizione prevista non è documentata.
- **Bilanciamento:** se l'assegnazione è casuale, i gruppi hanno le stesse caratteristiche precedenti al test (archetipo, dispositivo, canale, paese). Una V di Cramér vicina a zero lo conferma, anche dove l'SRM non si può verificare.

In [ ]:
lunghezze_diverse = con.sql("""
SELECT COUNT(*) FROM sessioni
WHERE enrolled_experiments IS NOT NULL
  AND len(string_split(enrolled_experiments, '|')) <> len(string_split(enrolled_variants, '|'))
""").fetchone()[0]
print("Sessioni con liste di esperimenti e varianti di lunghezza diversa:", num(lunghezze_diverse), "\n")

con.sql("""
CREATE OR REPLACE TABLE assegnazioni AS
SELECT session_id, user_id, session_start, had_purchase, ricavi,
       customer_segment, user_archetype, primary_device, device_type, traffic_source, country,
       UNNEST(esperimenti) AS esperimento,
       UNNEST(varianti) AS variante
FROM (
  SELECT *,
         string_split(enrolled_experiments, '|') AS esperimenti,
         string_split(enrolled_variants, '|') AS varianti
  FROM sessioni
  WHERE enrolled_experiments IS NOT NULL
)
""")

panoramica = q("""
SELECT esperimento, variante, COUNT(*) AS sessioni, COUNT(DISTINCT user_id) AS utenti,
       MIN(session_start)::DATE AS dal, MAX(session_start)::DATE AS al
FROM assegnazioni GROUP BY 1, 2 ORDER BY 1, 2
""")
display(panoramica)

coerenza = q("""
SELECT esperimento, COUNT(*) AS utenti,
       AVG((varianti > 1)::INTEGER) AS quota_utenti_in_piu_varianti
FROM (SELECT esperimento, user_id, COUNT(DISTINCT variante) AS varianti
      FROM assegnazioni GROUP BY 1, 2)
GROUP BY 1 ORDER BY 1
""")
display(coerenza)

UNITA = "user_id" if coerenza["quota_utenti_in_piu_varianti"].max() < 0.01 else "session_id"
DISPOSITIVO = "primary_device" if UNITA == "user_id" else "device_type"
print("Unità di analisi:", "l'utente" if UNITA == "user_id" else "la sessione")

In [ ]:
# Una riga per esperimento e unità: conversione = almeno un acquisto nelle sessioni dell'esperimento
con.sql(f"""
CREATE OR REPLACE TABLE unita_test AS
SELECT esperimento, {UNITA} AS unita, MIN(variante) AS variante,
       MAX(had_purchase::INTEGER) AS conversione,
       SUM(ricavi) AS ricavi,
       MIN(session_start) AS primo_ingresso,
       ANY_VALUE(customer_segment) AS segmento_cliente,
       ANY_VALUE(user_archetype) AS archetipo,
       ANY_VALUE({DISPOSITIVO}) AS dispositivo,
       ANY_VALUE(traffic_source) AS canale,
       ANY_VALUE(country) AS paese
FROM assegnazioni
GROUP BY esperimento, {UNITA}
HAVING COUNT(DISTINCT variante) = 1
""")

ris = q("""
SELECT esperimento, variante, COUNT(*) AS n, SUM(conversione) AS conversioni,
       AVG(ricavi) AS ricavi_medi, STDDEV_SAMP(ricavi) AS ricavi_sd,
       MIN(primo_ingresso)::DATE AS dal, MAX(primo_ingresso)::DATE AS al
FROM unita_test GROUP BY 1, 2 ORDER BY 1, 2
""")
ris["tasso"] = ris["conversioni"] / ris["n"]


def controllo_di(varianti):
    varianti = list(varianti)
    return "control" if "control" in varianti else sorted(varianti)[0]


srm = []
for esperimento, g in ris.groupby("esperimento"):
    riga = {"esperimento": esperimento, "varianti": " / ".join(g["variante"]),
            "unità": " / ".join(num(x) for x in g["n"]),
            "quote": " / ".join(pct(x, 0) for x in g["n"] / g["n"].sum())}
    if len(g) == 2:
        _, p = stats.chisquare(g["n"])
        riga["p SRM (50/50)"] = f"{p:.3g}"
        riga["esito"] = "OK" if p >= 0.001 else "Squilibrio: controllare l'assegnazione"
    else:
        riga["p SRM (50/50)"] = "–"
        riga["esito"] = "Ripartizione prevista non documentata: vedi bilanciamento"
    srm.append(riga)
display(pd.DataFrame(srm))

In [ ]:
# Bilanciamento: le varianti hanno le stesse caratteristiche precedenti al test?
COVARIATE = {"archetipo": "Archetipo", "dispositivo": "Dispositivo", "canale": "Canale", "paese": "Paese"}
bilanciamento = []
for esperimento in ris["esperimento"].unique():
    riga = {"esperimento": esperimento}
    for col, etichetta in COVARIATE.items():
        d = q(f"""
        SELECT variante, {col} AS livello, COUNT(*) AS n
        FROM unita_test WHERE esperimento = '{esperimento}'
        GROUP BY 1, 2
        """)
        tabella = d.pivot(index="variante", columns="livello", values="n").fillna(0)
        chi2, p, _, _ = stats.chi2_contingency(tabella.values)
        v = np.sqrt(chi2 / (tabella.values.sum() * (min(tabella.shape) - 1)))
        riga[etichetta] = f"V {v:.3f} · p {p:.2g}"
    bilanciamento.append(riga)
print("Associazione tra variante e caratteristiche dell'utente (V di Cramér vicina a 0 = gruppi simili)")
display(pd.DataFrame(bilanciamento))

### Potenza statistica

Per ogni esperimento: il più piccolo effetto che il test poteva rilevare con potenza dell'80% e confidenza del 95% (*minimum detectable effect*, MDE), e quante unità per gruppo servirebbero per rilevare un aumento del 5% della conversione, o per confermare il lift osservato quando è positivo. Un risultato non significativo con una MDE grande non vuol dire "nessun effetto": vuol dire che il test era troppo piccolo per vederlo.

La durata in giorni qui non serve: quasi tutti gli utenti entrano negli esperimenti nelle prime settimane, perché sono già clienti attivi. Per avere più unità bisogna coinvolgere più utenti, non aspettare di più.

In [ ]:
confronti = []
for esperimento, g in ris.groupby("esperimento"):
    g = g.set_index("variante")
    base = controllo_di(g.index)
    c = g.loc[base]
    for variante in g.index.drop(base):
        t = g.loc[variante]
        pc, pt = c["tasso"], t["tasso"]
        se = np.sqrt(pc * (1 - pc) / c["n"] + pt * (1 - pt) / t["n"])
        pool = (c["conversioni"] + t["conversioni"]) / (c["n"] + t["n"])
        z = (pt - pc) / np.sqrt(pool * (1 - pool) * (1 / c["n"] + 1 / t["n"]))
        welch = stats.ttest_ind_from_stats(t["ricavi_medi"], t["ricavi_sd"], t["n"],
                                           c["ricavi_medi"], c["ricavi_sd"], c["n"], equal_var=False)
        confronti.append({
            "esperimento": esperimento, "variante": variante, "controllo": base,
            "n_controllo": c["n"], "n_variante": t["n"],
            "conv_controllo": c["conversioni"], "conv_variante": t["conversioni"],
            "tasso_controllo": pc, "tasso_variante": pt,
            "diff": pt - pc, "diff_basso": pt - pc - Z * se, "diff_alto": pt - pc + Z * se,
            "p": 2 * stats.norm.sf(abs(z)),
            "mde": (Z + Z_POT) * np.sqrt(pc * (1 - pc) * (1 / c["n"] + 1 / t["n"])),
            "ricavi_controllo": c["ricavi_medi"], "ricavi_variante": t["ricavi_medi"], "p_ricavi": welch.pvalue,
        })
conf = pd.DataFrame(confronti)
conf["p_holm"] = holm(conf["p"])
for col in ["", "_basso", "_alto"]:
    conf["lift" + col] = conf["diff" + col] / conf["tasso_controllo"]
conf["mde_rel"] = conf["mde"] / conf["tasso_controllo"]


def unita_necessarie(pc, pt):
    return np.inf if pt == pc else (Z + Z_POT) ** 2 * (pc * (1 - pc) + pt * (1 - pt)) / (pt - pc) ** 2


conf["n_per_5pct"] = [unita_necessarie(pc, pc * 1.05) for pc in conf["tasso_controllo"]]
conf["n_per_osservato"] = [unita_necessarie(pc, pt) if pt > pc else np.nan
                           for pc, pt in zip(conf["tasso_controllo"], conf["tasso_variante"])]

display(pd.DataFrame({
    "esperimento": conf["esperimento"], "variante": conf["variante"],
    "unità controllo": conf["n_controllo"].map(num),
    "unità variante": conf["n_variante"].map(num),
    "conversione controllo": conf["tasso_controllo"].map(lambda x: pct(x, 2)),
    "MDE assoluta (punti)": (conf["mde"] * 100).map(lambda x: num(x, 2)),
    "MDE relativa": conf["mde_rel"].map(pct),
    "unità per gruppo per +5%": conf["n_per_5pct"].map(num),
    "unità per gruppo per confermare il lift osservato": conf["n_per_osservato"].map(num),
}))

### Risultati: lettura frequentista

Differenza di conversione tra variante e controllo con intervallo di confidenza al 95%, e p-value del test z su due proporzioni. Con sei esperimenti analizzati insieme, la probabilità di trovare almeno un falso positivo sale: i p-value sono corretti con il metodo di **Holm**. I ricavi per unità sono una metrica secondaria (test t di Welch).

In [ ]:
display(pd.DataFrame({
    "esperimento": conf["esperimento"], "variante": conf["variante"],
    "conv. controllo": conf["tasso_controllo"].map(lambda x: pct(x, 2)),
    "conv. variante": conf["tasso_variante"].map(lambda x: pct(x, 2)),
    "lift": conf["lift"].map(pct),
    "IC 95% lift": [f"{pct(b)} – {pct(a)}" for b, a in zip(conf["lift_basso"], conf["lift_alto"])],
    "p": conf["p"].map(lambda x: f"{x:.2g}"),
    "p Holm": conf["p_holm"].map(lambda x: f"{x:.2g}"),
    "ricavi/unità contr.": conf["ricavi_controllo"].map(lambda x: num(x, 2)),
    "ricavi/unità var.": conf["ricavi_variante"].map(lambda x: num(x, 2)),
    "p ricavi": conf["p_ricavi"].map(lambda x: f"{x:.2g}"),
}))

### Risultati: lettura bayesiana

Con una distribuzione a priori uniforme, la conversione di ogni variante ha una distribuzione Beta a posteriori. Da 200.000 estrazioni si ottengono tre numeri che il management legge più facilmente del p-value:

- **probabilità che la variante sia migliore** del controllo;
- **lift atteso** con intervallo di credibilità al 95%;
- **perdita attesa** se si adotta la variante e invece era peggiore, in percentuale della conversione attuale. Una regola tipica: adottare quando la probabilità supera il 95% e la perdita attesa è trascurabile.

In [ ]:
rng = np.random.default_rng(42)
ESTRAZIONI = 200_000
bayes = []
for r in conf.itertuples(index=False):
    a = rng.beta(1 + r.conv_controllo, 1 + r.n_controllo - r.conv_controllo, ESTRAZIONI)
    b = rng.beta(1 + r.conv_variante, 1 + r.n_variante - r.conv_variante, ESTRAZIONI)
    lift = b / a - 1
    bayes.append({"esperimento": r.esperimento, "variante": r.variante,
                  "P(variante migliore)": (b > a).mean(),
                  "lift atteso": np.median(lift),
                  "basso": np.quantile(lift, 0.025), "alto": np.quantile(lift, 0.975),
                  "perdita attesa": np.maximum(a - b, 0).mean() / r.tasso_controllo})
bayes = pd.DataFrame(bayes)
display(pd.DataFrame({
    "esperimento": bayes["esperimento"], "variante": bayes["variante"],
    "P(variante migliore)": bayes["P(variante migliore)"].map(pct),
    "lift atteso": bayes["lift atteso"].map(pct),
    "intervallo di credibilità 95%": [f"{pct(b)} – {pct(a)}" for b, a in zip(bayes["basso"], bayes["alto"])],
    "perdita attesa se si adotta": bayes["perdita attesa"].map(lambda x: pct(x, 2)),
}))

### Decisione per esperimento

La regola combina le due letture:

- **Adottare** o **non adottare** solo se l'effetto è significativo dopo la correzione di Holm.
- **Promettente** se non è significativo ma la probabilità bayesiana che la variante sia migliore supera il 95%: va confermato con un test dedicato, dimensionato sul lift osservato. Con dodici confronti, uno sopra il 95% può uscire anche per caso.
- Altrimenti **nessun effetto rilevabile**, con la MDE che dice quanto grande avrebbe dovuto essere per vederlo.

In [ ]:
conf["P_migliore"] = bayes["P(variante migliore)"].values


def decisione(r):
    if r["p_holm"] < 0.05:
        return "Adottare" if r["diff"] > 0 else "Non adottare: peggiora"
    if r["P_migliore"] >= 0.95:
        return f"Promettente: confermare con circa {num(r['n_per_osservato'], 0)} unità per gruppo"
    if r["P_migliore"] <= 0.05:
        return "Probabilmente peggiora: non adottare"
    return f"Nessun effetto rilevabile (MDE {pct(r['mde_rel'])})"


conf["decisione"] = conf.apply(decisione, axis=1)
display(pd.DataFrame({
    "esperimento": conf["esperimento"], "variante": conf["variante"],
    "lift": conf["lift"].map(pct), "p Holm": conf["p_holm"].map(lambda x: f"{x:.2g}"),
    "P(variante migliore)": conf["P_migliore"].map(pct), "decisione": conf["decisione"],
}))

In [ ]:
ordine = conf.sort_values("lift").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 0.55 * len(ordine) + 1.6))
y = np.arange(len(ordine))
ax.errorbar(ordine["lift"] * 100, y,
            xerr=[(ordine["lift"] - ordine["lift_basso"]) * 100, (ordine["lift_alto"] - ordine["lift"]) * 100],
            fmt="none", ecolor=GRIGIO, elinewidth=3)
ax.scatter(ordine["lift"] * 100, y, zorder=3, s=40,
           color=[ACCENTO if p < 0.05 else SCURO for p in ordine["p_holm"]])
ax.axvline(0, color="#333333", lw=1)
ax.set_yticks(y, [f"{e} · {v}" for e, v in zip(ordine["esperimento"], ordine["variante"])])
ax.xaxis.set_major_formatter(lambda x, _: f"{x:+.0f}%")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Lift della conversione per esperimento, con IC 95% (in blu: significativo dopo la correzione di Holm)",
             loc="left", fontsize=11)
plt.tight_layout()
plt.show()

### Per chi funziona: effetti per segmento

Un effetto medio può nascondere segmenti che reagiscono in modo opposto. Per ogni esperimento si stima il lift per segmento e si verifica con il **test Q di Cochran** se le differenze tra segmenti sono più grandi di quanto ci si aspetterebbe dal caso. Un p basso indica un effetto eterogeneo: la decisione può cambiare da segmento a segmento. Anche qui i test sono molti, quindi i p-value sono corretti con Holm.

Si segmenta solo per caratteristiche fissate **prima** del test. Una variabile calcolata sugli acquisti, come un segmento di valore del cliente, dipende dal risultato del test stesso e lo falserebbe. La cella seguente lo verifica: se un livello ha conversione 0% o 100%, la variabile è costruita sugli acquisti e si esclude.

In [ ]:
CANDIDATE = {"segmento_cliente": "Segmento cliente", "archetipo": "Archetipo",
             "dispositivo": "Dispositivo", "canale": "Canale", "paese": "Paese"}
SEGMENTI = {}
for col, etichetta in CANDIDATE.items():
    d = q(f"""
    SELECT {col} AS livello, COUNT(*) AS unita, AVG(conversione) AS conversione
    FROM unita_test GROUP BY 1 ORDER BY 3
    """)
    costruita_sugli_acquisti = (d["conversione"].min() == 0) or (d["conversione"].max() == 1)
    if not costruita_sugli_acquisti:
        SEGMENTI[col] = etichetta
    print(f"\n{etichetta}: " + ("ESCLUSA, dipende dagli acquisti" if costruita_sugli_acquisti else "usabile"))
    print("  " + " · ".join(f"{l} {pct(c)}" for l, c in zip(d["livello"], d["conversione"])))

In [ ]:
def per_segmento(esperimento, dim):
    d = q(f"""
    SELECT {dim} AS segmento, variante, COUNT(*) AS n, SUM(conversione) AS conversioni
    FROM unita_test WHERE esperimento = '{esperimento}'
    GROUP BY 1, 2
    """)
    base = controllo_di(d["variante"].unique())
    righe = []
    for segmento, g in d.groupby("segmento"):
        g = g.set_index("variante")
        if base not in g.index:
            continue
        c = g.loc[base]
        for variante in g.index.drop(base):
            t = g.loc[variante]
            pc, pt = c["conversioni"] / c["n"], t["conversioni"] / t["n"]
            se = np.sqrt(pc * (1 - pc) / c["n"] + pt * (1 - pt) / t["n"])
            # log odds ratio, con 0,5 aggiunto alle celle per evitare divisioni per zero
            a1, b1 = t["conversioni"] + 0.5, t["n"] - t["conversioni"] + 0.5
            a0, b0 = c["conversioni"] + 0.5, c["n"] - c["conversioni"] + 0.5
            righe.append({"segmento": segmento, "variante": variante, "n": c["n"] + t["n"],
                          "tasso_controllo": pc, "tasso_variante": pt,
                          "lift": (pt - pc) / pc if pc > 0 else np.nan,
                          "lift_basso": (pt - pc - Z * se) / pc if pc > 0 else np.nan,
                          "lift_alto": (pt - pc + Z * se) / pc if pc > 0 else np.nan,
                          "log_or": np.log(a1 * b0 / (b1 * a0)),
                          "var_log_or": 1 / a1 + 1 / b1 + 1 / a0 + 1 / b0})
    return pd.DataFrame(righe)


def cochran_q(seg):
    w = 1 / seg["var_log_or"]
    media = (w * seg["log_or"]).sum() / w.sum()
    q_stat = (w * (seg["log_or"] - media) ** 2).sum()
    return q_stat, stats.chi2.sf(q_stat, len(seg) - 1)


eterogeneita, dettagli = [], {}
for r in conf.itertuples(index=False):
    for dim, etichetta in SEGMENTI.items():
        seg = per_segmento(r.esperimento, dim)
        seg = seg[seg["variante"] == r.variante].reset_index(drop=True)
        if len(seg) < 2:
            continue
        dettagli[(r.esperimento, r.variante, dim)] = seg
        q_stat, p = cochran_q(seg)
        alto, basso = seg.loc[seg["lift"].idxmax()], seg.loc[seg["lift"].idxmin()]
        eterogeneita.append({"esperimento": r.esperimento, "variante": r.variante, "dimensione": etichetta,
                             "Q": round(q_stat, 1), "p": p,
                             "lift più alto": f"{alto['segmento']} {pct(alto['lift'])}",
                             "lift più basso": f"{basso['segmento']} {pct(basso['lift'])}"})
eterogeneita = pd.DataFrame(eterogeneita)
eterogeneita["p_holm"] = holm(eterogeneita["p"])
print(f"{len(eterogeneita)} test di eterogeneità; con p < 0,05 se ne aspettano circa {len(eterogeneita) * 0.05:.0f} per caso")
display(eterogeneita.assign(p=eterogeneita["p"].map(lambda x: f"{x:.2g}"),
                            p_holm=eterogeneita["p_holm"].map(lambda x: f"{x:.2g}")))

In [ ]:
# Dettaglio per l'esperimento con l'effetto più netto, sulla dimensione più eterogenea
principale = conf.sort_values("p").iloc[0]
ESP, VAR = principale["esperimento"], principale["variante"]
righe_esp = eterogeneita[(eterogeneita["esperimento"] == ESP) & (eterogeneita["variante"] == VAR)]
dim_label = righe_esp.sort_values("p").iloc[0]["dimensione"]
DIM = {v: k for k, v in SEGMENTI.items()}[dim_label]
seg = dettagli[(ESP, VAR, DIM)].sort_values("lift").reset_index(drop=True)

print(f"{ESP} · {VAR}: lift medio {pct(principale['lift'])}, per {dim_label.lower()}")
display(pd.DataFrame({
    dim_label: seg["segmento"], "unità": seg["n"].map(num),
    "conv. controllo": seg["tasso_controllo"].map(lambda x: pct(x, 2)),
    "conv. variante": seg["tasso_variante"].map(lambda x: pct(x, 2)),
    "lift": seg["lift"].map(pct),
    "IC 95%": [f"{pct(b)} – {pct(a)}" for b, a in zip(seg["lift_basso"], seg["lift_alto"])],
}))

fig, ax = plt.subplots(figsize=(10, 0.5 * len(seg) + 1.6))
y = np.arange(len(seg))
ax.errorbar(seg["lift"] * 100, y,
            xerr=[(seg["lift"] - seg["lift_basso"]) * 100, (seg["lift_alto"] - seg["lift"]) * 100],
            fmt="none", ecolor=GRIGIO, elinewidth=3)
ax.scatter(seg["lift"] * 100, y, color=SCURO, zorder=3, s=40)
ax.axvline(principale["lift"] * 100, color=ACCENTO, lw=1.5, ls="--", label="lift medio")
ax.axvline(0, color="#333333", lw=1)
ax.set_yticks(y, seg["segmento"])
ax.xaxis.set_major_formatter(lambda x, _: f"{x:+.0f}%")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
ax.set_title(f"{ESP}: lift della conversione per {dim_label.lower()}, con IC 95%", loc="left", fontsize=11)
plt.tight_layout()
plt.show()

### Stabilità nel tempo

Un effetto che cala trimestre dopo trimestre può essere un *effetto novità*: gli utenti reagiscono alla variante perché è nuova, non perché è migliore. Si controlla sull'esperimento principale: per ogni trimestre, la conversione degli utenti attivi in quel trimestre. Il trimestre di ingresso non servirebbe, perché quasi tutti entrano nel primo.

In [ ]:
tempo = q(f"""
WITH attivi AS (
  SELECT date_trunc('quarter', session_start)::DATE AS trimestre, user_id, variante,
         MAX(had_purchase::INTEGER) AS conversione
  FROM assegnazioni WHERE esperimento = '{ESP}'
  GROUP BY 1, 2, 3
)
SELECT trimestre, variante, COUNT(*) AS n, SUM(conversione) AS conversioni
FROM attivi GROUP BY 1, 2 ORDER BY 1, 2
""")
righe = []
for trimestre, g in tempo.groupby("trimestre"):
    g = g.set_index("variante")
    if principale["controllo"] not in g.index or VAR not in g.index:
        continue
    c, t = g.loc[principale["controllo"]], g.loc[VAR]
    if min(c["n"], t["n"]) < 500:
        continue
    pc, pt = c["conversioni"] / c["n"], t["conversioni"] / t["n"]
    se = np.sqrt(pc * (1 - pc) / c["n"] + pt * (1 - pt) / t["n"])
    righe.append({"trimestre": pd.Timestamp(trimestre).to_period("Q").strftime("%Y-T%q"),
                  "unità": num(c["n"] + t["n"]),
                  "lift": pct((pt - pc) / pc),
                  "IC 95%": f"{pct((pt - pc - Z * se) / pc)} – {pct((pt - pc + Z * se) / pc)}"})
print(f"{ESP} · {VAR}: lift per trimestre, utenti attivi nel trimestre (esclusi i trimestri con meno di 500 utenti per gruppo)")
display(pd.DataFrame(righe))

## 6 · Lettura dei risultati

Esecuzione dell'8 novembre 2026 su 5.651.586 sessioni del 2023–2024, esclusi i bot.

**Funnel (domanda 2).** Su 100 sessioni, 3,2 aggiungono un prodotto al carrello, 1,1 avviano il checkout e 0,8 comprano. Canale, browser e dispositivo non cambiano nulla: V di Cramér sotto 0,01 in ogni passaggio. L'unica differenza che conta è **Flipkart**: arriva al checkout il 15,9% dei carrelli contro il 37,9% delle altre piattaforme (V 0,17), e la conversione è 0,37% contro circa 0,88%. Il calo è lo stesso su ogni dispositivo (15,5–16,3%), canale (15,5–16,5%), paese (13,9–17,1%) e trimestre dal 2023 (15,2–16,2%). Non riguarda quindi un segmento né un periodo: è il passaggio al checkout di Flipkart. Con il tasso delle altre piattaforme, Flipkart avrebbe fatto 4.634 acquisti e $849.618 di ricavi in più in due anni, circa $425.000 l'anno, più di quanto ha incassato ($612.778).

**Attribution (domanda 1).** Non c'è un segnale da attribuire. Ogni utente arriva sempre dallo stesso canale: il 100% dei percorsi ha un solo canale. La conversione è la stessa per tutti i canali, tra 0,78% e 0,84%, con intervalli che si sovrappongono. Così i cinque modelli, Markov compreso, coincidono con la quota di traffico, con scarti sotto 0,2 punti. È la stessa conclusione di theLook: un modello non può trovare un segnale che i dati non contengono. Per misurare il valore dei canali servono dati con più contatti per utente e, soprattutto, test di incrementalità, per esempio sospendendo un canale in alcune aree.

**A/B test (domanda 5).**

- **Il disegno regge.** Ogni utente resta sempre nella stessa variante, quindi l'unità di analisi è l'utente. I tre esperimenti a due varianti sono divisi a metà, senza sample ratio mismatch. I tre a quattro varianti hanno ripartizioni non documentate, ma in tutti i gruppi archetipo, dispositivo, canale e paese sono distribuiti allo stesso modo (V di Cramér al massimo 0,014): l'assegnazione è casuale.
- **Nessuna variante ha un effetto significativo** dopo la correzione di Holm, né sulla conversione né sui ricavi per utente.
- **Il motivo principale è la potenza.** Con circa 50.000 utenti per gruppo, i test potevano vedere solo effetti dal 6% in su, mentre un cambio di interfaccia vale di solito pochi punti percentuali. Per vedere un +5% servono circa 70.000 utenti per gruppo, 1,4 volte quelli usati. Le varianti più piccole (`treatment_full` e `treatment_low`) avevano tra 1.300 e 5.300 utenti, con MDE tra il 13% e il 26%.
- **Il segnale più forte è `treatment_full` di `platform_price_comparison`:** +9,2% di conversione (IC da −0,6% a +18,9%), 97% di probabilità di essere migliore del controllo, perdita attesa 0,05%. Con dodici confronti, uno sopra il 95% può uscire anche per caso. Va confermato con un test dedicato, controllo contro `treatment_full`, con circa 21.600 utenti per gruppo, quattro volte quelli attuali.
- **L'effetto non cambia da un segmento all'altro.** Su 48 test di eterogeneità nessuno resta significativo dopo la correzione di Holm: il p più basso, 0,0027, diventa 0,13. Il +56% di `treatment_full` in Germania è il classico falso positivo da sottogruppo: un paese su undici, con un test di eterogeneità a p 0,45.
- **Nessun effetto novità.** Il lift di `treatment_full` oscilla tra −18% e +40% da un trimestre all'altro, con intervalli larghi e senza una tendenza al calo.
- **Una variabile da non usare.** Il segmento cliente è costruito sugli acquisti: `high_value` converte al 100%, `loyal` allo 0%. Segmentare un test con una variabile così ne falsa il risultato. Il notebook la riconosce e la esclude da solo.

**Raccomandazioni**

1. **Flipkart:** ricostruire il passaggio dal carrello al checkout (pagine, passaggi richiesti, metodi di pagamento) e testare una correzione. Vale circa $425.000 l'anno.
2. **Esperimenti:** fissare prima dell'avvio l'effetto minimo che interessa e il numero di utenti necessario. Meglio meno esperimenti in parallelo, ciascuno abbastanza grande da vedere effetti del 3–5%.
3. **Confronto prezzi:** confermare `treatment_full` con un test dedicato prima di adottarlo.
4. **Attribution:** prima di scegliere un modello, raccogliere dati con più contatti per utente e misurare l'incrementalità con test di sospensione.

I dati sono sintetici: i risultati mostrano il metodo, e alcuni effetti, come il calo di Flipkart, possono essere stati scritti nel generatore.

## 7 · Dati per la dashboard Tableau Public

Un file Excel con un foglio per ogni vista della dashboard. I numeri sono già aggregati: Tableau non deve leggere i 5,7 milioni di sessioni.

- `funnel`: una riga per piattaforma e fase, con le quote sul totale e sulla fase precedente;
- `flipkart_dettaglio`: il passaggio dal carrello al checkout della piattaforma peggiore e delle altre, per dispositivo, canale, paese e trimestre;
- `canali`: conversione per canale e quota di conversioni secondo i cinque modelli di attribution;
- `esperimenti`: un riga per variante, con lift, intervallo di confidenza, p-value, probabilità bayesiana e decisione.

Il file si scarica da solo alla fine dell'esecuzione. Se il browser lo blocca, è nel pannello *File* di Colab, a sinistra, nella cartella `content`.

In [ ]:
FASI = [("Sessioni", "sessioni"), ("Carrello", "carrello"), ("Checkout", "checkout"), ("Acquisto", "acquisto")]
righe = []
for _, r in funnel_dim["platform"].iterrows():
    precedente = None
    for ordine, (fase, col) in enumerate(FASI, start=1):
        n = float(r[col])
        righe.append({"piattaforma": r["valore"],
                      "gruppo": r["valore"] if r["valore"] == PEGGIORE else "Altre piattaforme",
                      "ordine_fase": ordine, "fase": fase, "sessioni": int(n),
                      "quota_sul_totale": n / float(r["sessioni"]),
                      "quota_sulla_fase_precedente": n / precedente if precedente else None})
        precedente = n
foglio_funnel = pd.DataFrame(righe)

righe = []
for espressione, etichetta in SCOMPOSIZIONI.items():
    d = q(f"""
    SELECT {espressione} AS valore,
           SUM(CASE WHEN platform = '{PEGGIORE}' THEN 1 ELSE 0 END) AS carrelli_peggiore,
           SUM(CASE WHEN platform = '{PEGGIORE}' THEN checkout_started::INTEGER ELSE 0 END) AS checkout_peggiore,
           SUM(CASE WHEN platform <> '{PEGGIORE}' THEN 1 ELSE 0 END) AS carrelli_altre,
           SUM(CASE WHEN platform <> '{PEGGIORE}' THEN checkout_started::INTEGER ELSE 0 END) AS checkout_altre
    FROM sessioni WHERE had_cart_event
    GROUP BY 1 ORDER BY 1
    """)
    for _, r in d.iterrows():
        valore = r["valore"]
        if etichetta == "Trimestre":
            valore = pd.Timestamp(valore).to_period("Q").strftime("%Y-T%q")
        for gruppo, suffisso in [(PEGGIORE, "peggiore"), ("Altre piattaforme", "altre")]:
            carrelli, checkout = float(r["carrelli_" + suffisso]), float(r["checkout_" + suffisso])
            righe.append({"dimensione": etichetta, "valore": str(valore), "gruppo": gruppo,
                          "carrelli": int(carrelli), "checkout": int(checkout),
                          "checkout_su_carrello": checkout / carrelli if carrelli else None})
foglio_dettaglio = pd.DataFrame(righe)

conv_canali = funnel_dim["traffic_source"].rename(columns={"valore": "canale"})
basso, alto = wilson(conv_canali["acquisto"], conv_canali["sessioni"])
foglio_canali = (attribution.reset_index()
                 .rename(columns={"Quota di sessioni": "quota_sessioni", "Last click": "last_click",
                                  "First click": "first_click", "Lineare": "lineare", "A U": "a_u",
                                  "Markov": "markov"})
                 .merge(pd.DataFrame({"canale": conv_canali["canale"],
                                      "acquisti": conv_canali["acquisto"].astype(int),
                                      "conversione": conv_canali["Conversione totale"],
                                      "conversione_ic_basso": basso, "conversione_ic_alto": alto}),
                        on="canale"))
foglio_canali = foglio_canali[["canale", "sessioni", "acquisti", "conversione", "conversione_ic_basso",
                               "conversione_ic_alto", "quota_sessioni", "last_click", "first_click",
                               "lineare", "a_u", "markov"]]

foglio_esperimenti = conf[["esperimento", "variante", "n_controllo", "n_variante", "tasso_controllo",
                           "tasso_variante", "lift", "lift_basso", "lift_alto", "p", "p_holm",
                           "P_migliore", "mde_rel", "n_per_osservato", "decisione"]].copy()
foglio_esperimenti.insert(0, "etichetta", foglio_esperimenti["esperimento"] + " · " + foglio_esperimenti["variante"])
foglio_esperimenti["esito"] = (foglio_esperimenti["decisione"].str.split(":").str[0]
                               .str.replace(r" \(MDE.*", "", regex=True))
foglio_esperimenti = foglio_esperimenti.rename(columns={"P_migliore": "prob_variante_migliore",
                                                        "n_per_osservato": "utenti_per_confermare"})

FILE_TABLEAU = "/content/tableau_funnel_esperimenti.xlsx"
fogli = {"funnel": foglio_funnel, "flipkart_dettaglio": foglio_dettaglio,
         "canali": foglio_canali, "esperimenti": foglio_esperimenti}
with pd.ExcelWriter(FILE_TABLEAU) as writer:
    for nome, foglio in fogli.items():
        foglio.to_excel(writer, sheet_name=nome, index=False)
for nome, foglio in fogli.items():
    print(f"{nome:20s} {len(foglio):4d} righe · {', '.join(foglio.columns)}")

try:
    from google.colab import files
    files.download(FILE_TABLEAU)
except ImportError:
    print("File salvato in", FILE_TABLEAU)